In [1]:
import os
import time
import pandas as pd
from ytmusicapi import YTMusic

PLAYLIST_LIMIT=500
PLAYLIST_SONG_LIMIT=10000
yt = YTMusic('../headers_auth.json')

def parse_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])  # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_playlist(yt, playlist_meta, print_meta=False):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    if print_meta: 
        print(pd.DataFrame.from_dict(playlist_meta, orient='index'))
    tracks = parse_tracks(track_list)
    return tracks, playlist_meta

def create_rating_playlist_subset(tracks, name, rating):
    assert rating in ('LIKE', 'DISLIKE', 'INDIFFERENT')
    filtered_tracks = tracks.loc[tracks['likeStatus'] == rating]
    video_ids = filtered_tracks['videoId'].unique().tolist()
    pl_id = yt.create_playlist(
        title=name + ' ' + rating.lower(), 
        description='generated from %s includes %s subset' % (name, rating),
        privacy_status='PRIVATE', 
        video_ids=video_ids
    )
    print('Created %s playlist with id %s' % (rating, pl_id))

In [2]:
%%time
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
print('Playlists:\n %s' % sorted(playlists.sort_values('title')['title']))

Playlists:
 ['1950s_thumbs_up', '1960s_thumbs_up', '1970s_thumbs_up', '1980s_thumbs_up', '1990s_thumbs_up', '2000s_thumbs_up', '2005s_thumbs_up', '2010_thumbs_up', '2010s Electronic like', '2011_thumbs_up', '2012_thumbs_up', '2013_thumbs_up', '2014_thumbs_up', '2015_thumbs_up', '2016_thumbs_up', '2017_thumbs_up', '2018_thumbs_up', '2018_top_50_albums', '2019_thumbs_up', '2019_top_50_albums', 'Ambient Unrated Albums 2018-2019', 'Analog Grooves like', 'Beat instrumentals', 'Brass n chill', "California Dreamin' indifferent", "California Dreamin' like", 'Chill Indie Beats indifferent', 'Chill Indie Beats like', 'Chillwave', "Classic Rock's Greatest Hits indifferent", "Classic Rock's Greatest Hits like", 'Classic West Coast Hip Hop indifferent', 'Classic West Coast Hip Hop like', 'Classical piano', 'Dance radio', 'Deep Cut Kick Back indifferent', 'Deep Cut Kick Back like', 'Electronic Focus indifferent', 'Electronic Focus like', 'Essential New Wave indifferent', 'Essential New Wave like', '

In [3]:
# # Example: Create Unrated and Liked Subset Playlist

# playlist_name = 'Analog Grooves'
# playlist = playlists.loc[playlists['title'] == playlist_name].iloc[0] # first match
# metadata = yt.get_playlist(playlist['playlistId'], limit=PLAYLIST_SONG_LIMIT)
# tracks, metadata = parse_playlist(yt, metadata)
# print('Selected Playlist:\n%s' % metadata)

# create_rating_playlist_subset(tracks, playlist_name, 'INDIFFERENT')
# create_rating_playlist_subset(tracks, playlist_name, 'LIKE')

In [16]:
%%time

# Example: Group public playlists
public_playlists = {}
privacy = 'PUBLIC'
for i, p in playlists.iterrows():
    if i == 0: continue # skip giant likes playlist
    metadata = yt.get_playlist(p['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    if metadata['privacy'] == privacy:
        public_playlists[p['playlistId']] = p['title']
        print('Found %s playlist named: %s' % (privacy.lower(), p['title']))

list(public_playlists.values())

Found public playlist named: classical guitar
Found public playlist named: Music Visuals
Found public playlist named: zz__ipod_albums
CPU times: user 43.7 s, sys: 2.89 s, total: 46.6 s
Wall time: 7min 54s


['classical guitar', 'Music Visuals', 'zz__ipod_albums']

In [3]:
%%time
# For each playlist, Create Unrated and Liked Subset Playlist, delete original
playlist_names = ["Deep Cut Kick Back indifferent"]
for playlist_name in playlist_names:
    print('Sorting %s in to like and indifferent playlists and deleting original' % playlist_name)
    playlist = playlists.loc[playlists['title'] == playlist_name].iloc[0] # first match
    metadata = yt.get_playlist(playlist['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    tracks, metadata = parse_playlist(yt, metadata)
    create_rating_playlist_subset(tracks, playlist_name, 'INDIFFERENT')
    time.sleep(20)
    create_rating_playlist_subset(tracks, playlist_name, 'LIKE')
    time.sleep(20)

Sorting Deep Cut Kick Back indifferent in to like and indifferent playlists and deleting original
Created INDIFFERENT playlist with id PLWptjpDqazOwBb_8uVr8IQgtblj0nQb8u
Created LIKE playlist with id PLWptjpDqazOxAOwXuirxEnytaXAcOBUql
CPU times: user 92.9 ms, sys: 9.56 ms, total: 103 ms
Wall time: 42 s
